# IndoBERT Production Training for Indonesian Toxic Speech

This notebook trains `indolem/indobertweet-base-uncased` directly for production use. It does not compare against baseline or alternate models.

Production flow:

1. Load and validate `dataset/indonesian_toxicspeech.csv`.
2. Split train/validation/test with stratification.
3. Fine-tune IndoBERTweet for binary toxic speech classification.
4. Tune the toxic-class decision threshold on validation data.
5. Report final test metrics once with the selected threshold.
6. Export Hugging Face, ONNX opset 18, and quantized ONNX Runtime artifacts.
7. Run CPU inference smoke tests with the production prediction helper.

The checkpoint warning about newly initialized `classifier.weight` and `classifier.bias` is expected: the base IndoBERTweet checkpoint does not contain this task-specific classification head. The training section below is the required downstream training step that makes that head usable for inference.


## Setup

Run this cell in Colab before training. It upgrades only the notebook-specific libraries and leaves Colab-managed core packages (`torch`, `pandas`, `numpy`, CUDA packages, plotting libraries) at the runtime versions to avoid dependency conflicts with `google-colab`, RAPIDS, `torchvision`, and `torchaudio`.

`onnxscript` is required by PyTorch's current Dynamo ONNX exporter. The notebook intentionally does not install or use Optimum, which avoids Optimum/diffusers export warnings and keeps ONNX export on the current PyTorch path.


In [ ]:
# Colab setup: avoid upgrading Colab-managed core packages such as torch, pandas, numpy, and CUDA libraries.
# Restart the runtime if Colab upgrades notebook-specific packages during install.
%pip install -q --upgrade --upgrade-strategy only-if-needed transformers datasets accelerate onnx onnxscript onnxruntime scikit-learn


In [ ]:
import importlib.metadata as importlib_metadata
import inspect
import json
import math
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "indolem/indobertweet-base-uncased"
MODEL_KEY = "indobertweet_production"
LABEL_MAPPING = {0: "non_toxic", 1: "toxic"}
ID2LABEL = {idx: label for idx, label in LABEL_MAPPING.items()}
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}

ARTIFACT_DIR = Path("artifacts")
OUTPUT_DIR = Path("outputs")
PT_DIR = ARTIFACT_DIR / "toxic_speech_model_pt"
ONNX_DIR = ARTIFACT_DIR / "toxic_speech_model_onnx"
QUANTIZED_DIR = ARTIFACT_DIR / "toxic_speech_model_onnx_quantized"

for path in [ARTIFACT_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
PACKAGE_DISTRIBUTIONS = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
    "onnxscript": "onnxscript",
    "scikit-learn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
}


def installed_package_versions():
    versions = {}
    for display_name, distribution_name in PACKAGE_DISTRIBUTIONS.items():
        try:
            versions[display_name] = importlib_metadata.version(distribution_name)
        except importlib_metadata.PackageNotFoundError:
            versions[display_name] = "not installed"
    return versions


print(f"Seed set to {SEED}")
for package_name, package_version in installed_package_versions().items():
    print(f"{package_name}: {package_version}")
print(f"Torch device: {DEVICE}")
if USE_CUDA:
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


## Dataset Loading and Validation

Input contract: a CSV with exactly `text` and `is_toxic`, where `is_toxic` is binary (`0 = non_toxic`, `1 = toxic`). By default, the notebook loads the dataset directly from the project GitHub raw URL. Text is only stripped for surrounding whitespace; slang, punctuation, informal words, and toxic terms are preserved because they can carry classification signal.


In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/BayuSatrio2804/Indonesia-Toxic-Speech-Detector/refs/heads/main/dataset/indonesian_toxicspeech.csv"

# Set DATASET_PATH manually if you want to use a local/uploaded CSV instead of the GitHub raw dataset.
DATASET_PATH = None
DEFAULT_DATASET_PATHS = [
    Path("dataset/indonesian_toxicspeech.csv"),
    Path("../dataset/indonesian_toxicspeech.csv"),
    Path("/content/indonesian_toxicspeech.csv"),
    Path("/content/drive/MyDrive/indonesian_toxicspeech.csv"),
]

if DATASET_PATH is not None:
    data_source = DATASET_PATH
elif DATASET_URL:
    data_source = DATASET_URL
else:
    data_source = next((candidate for candidate in DEFAULT_DATASET_PATHS if candidate.exists()), None)

if data_source is None:
    raise FileNotFoundError(
        "Dataset not found. Set DATASET_URL or DATASET_PATH to a CSV with text and is_toxic columns."
    )

raw_df = pd.read_csv(data_source)
print(f"Loaded {data_source} with shape {raw_df.shape}")
raw_df.head()


In [ ]:
EXPECTED_COLUMNS = {"text", "is_toxic"}
actual_columns = set(raw_df.columns)
if actual_columns != EXPECTED_COLUMNS:
    raise ValueError(f"Expected columns {EXPECTED_COLUMNS}, got {actual_columns}")

df = raw_df.copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].copy()
df["is_toxic"] = df["is_toxic"].astype(int)

labels = set(df["is_toxic"].unique().tolist())
if not labels.issubset({0, 1}):
    raise ValueError(f"Labels must be binary 0/1, got {sorted(labels)}")

word_lengths = df["text"].str.split().str.len()
print(f"Rows after validation: {len(df):,}")
print("Class distribution:")
print(df["is_toxic"].value_counts().sort_index().rename(index=LABEL_MAPPING))
print("Missing values:")
print(df.isna().sum())
print(f"Duplicate text count: {df['text'].duplicated().sum():,}")
print("Word length summary:")
print(word_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="is_toxic", ax=axes[0])
axes[0].set_title("Class Distribution")
axes[0].set_xticklabels([LABEL_MAPPING[int(tick.get_text())] for tick in axes[0].get_xticklabels()])
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Rows")

sns.histplot(word_lengths, bins=40, ax=axes[1])
axes[1].set_title("Text Length Distribution")
axes[1].set_xlabel("Words")
plt.tight_layout()


## Split

The split is fixed at `80/10/10` with stratification and seed `42`. Validation is used for threshold tuning. Test is touched once for final reporting after the threshold is selected.


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["is_toxic"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["is_toxic"],
)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    distribution = frame["is_toxic"].value_counts(normalize=True).sort_index().to_dict()
    print(f"{name}: {len(frame):,} rows, class ratios={distribution}")


## Metrics Helpers


In [ ]:
def extract_logits(outputs):
    if hasattr(outputs, "logits"):
        outputs = outputs.logits
    if isinstance(outputs, (tuple, list)):
        outputs = outputs[0]
    if hasattr(outputs, "detach"):
        outputs = outputs.detach().cpu().numpy()
    logits = np.asarray(outputs)
    if logits.ndim != 2 or logits.shape[1] < 2:
        raise ValueError(f"Expected logits with shape (n_examples, n_labels), got {logits.shape}")
    return logits


def softmax_np(logits):
    logits = extract_logits(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def binary_metrics(y_true, toxic_probs, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    toxic_probs = np.asarray(toxic_probs)
    y_pred = (toxic_probs >= threshold).astype(int)
    metrics = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    metrics["roc_auc"] = float(roc_auc_score(y_true, toxic_probs)) if len(np.unique(y_true)) == 2 else None
    return metrics


def print_metrics(title, metrics):
    print(title)
    for key in ["threshold", "accuracy", "precision", "recall", "f1", "roc_auc"]:
        value = metrics.get(key)
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
    print(f"  confusion_matrix: {metrics['confusion_matrix']}")


## IndoBERTweet Tokenization and Training

`classifier.weight` and `classifier.bias` are expected to be newly initialized when `AutoModelForSequenceClassification` loads this base checkpoint. The `cls.predictions.*` keys are the masked-language-model head from the pretraining checkpoint and are expected to be unused by sequence classification. The training cell below is the downstream training step that makes the new classification head usable for inference.

The `Trainer` call uses the current `processing_class=tokenizer` API instead of the deprecated `tokenizer=` argument, and warmup is configured with `warmup_steps` instead of deprecated `warmup_ratio`.


In [ ]:
MAX_LENGTH = int(min(256, max(64, math.ceil(word_lengths.quantile(0.95) / 16) * 16)))
MAX_LENGTH = max(MAX_LENGTH, 128)
print(f"Selected max sequence length: {MAX_LENGTH}")


def make_hf_dataset(frame):
    return Dataset.from_pandas(
        frame[["text", "is_toxic"]].rename(columns={"is_toxic": "labels"}).reset_index(drop=True),
        preserve_index=False,
    )


def align_model_special_tokens(model, tokenizer):
    for attr in ["pad_token_id", "bos_token_id", "eos_token_id"]:
        if hasattr(tokenizer, attr):
            token_id = getattr(tokenizer, attr)
            setattr(model.config, attr, token_id)
            if getattr(model, "generation_config", None) is not None:
                setattr(model.generation_config, attr, token_id)
    return model


def load_tokenizer_and_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    added_tokens = 0
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        added_tokens += 1

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    if added_tokens:
        model.resize_token_embeddings(len(tokenizer))
    return align_model_special_tokens(model, tokenizer), tokenizer


def tokenize_datasets(tokenizer, train_frame, val_frame, test_frame, max_length=128):
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    train_ds = make_hf_dataset(train_frame).map(tokenize_batch, batched=True, remove_columns=["text"])
    val_ds = make_hf_dataset(val_frame).map(tokenize_batch, batched=True, remove_columns=["text"])
    test_ds = make_hf_dataset(test_frame).map(tokenize_batch, batched=True, remove_columns=["text"])
    return train_ds, val_ds, test_ds


def compute_trainer_metrics(eval_pred):
    logits = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]
    probs = softmax_np(logits)[:, 1]
    return {key: value for key, value in binary_metrics(labels, probs, threshold=0.5).items() if key != "confusion_matrix"}


def make_training_args(output_dir, use_cuda, train_size):
    params = inspect.signature(TrainingArguments).parameters
    bf16_enabled = bool(use_cuda and torch.cuda.is_bf16_supported())
    fp16_enabled = bool(use_cuda and not bf16_enabled)
    train_batch_size = 16 if use_cuda else 8
    num_train_epochs = 3
    steps_per_epoch = math.ceil(train_size / train_batch_size)
    warmup_steps = max(1, round(steps_per_epoch * num_train_epochs * 0.06))
    common = {
        "output_dir": str(output_dir),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": train_batch_size,
        "per_device_eval_batch_size": 32 if use_cuda else 16,
        "num_train_epochs": num_train_epochs,
        "weight_decay": 0.01,
        "warmup_steps": warmup_steps,
        "logging_steps": 50,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
    }
    if "eval_strategy" in params:
        common["eval_strategy"] = "epoch"
    else:
        common["evaluation_strategy"] = "epoch"
    if "bf16" in params:
        common["bf16"] = bf16_enabled
    if "fp16" in params:
        common["fp16"] = fp16_enabled
    if "save_safetensors" in params:
        common["save_safetensors"] = True
    return TrainingArguments(**common)


def make_trainer(model, tokenizer, train_ds, val_ds):
    return Trainer(
        model=model,
        args=make_training_args(OUTPUT_DIR / MODEL_KEY, USE_CUDA, len(train_ds)),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_trainer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )


def predict_transformer_probs(trainer, dataset):
    predictions = trainer.predict(dataset)
    return softmax_np(predictions.predictions)[:, 1]


In [ ]:
model, tokenizer = load_tokenizer_and_model(MODEL_NAME)
train_ds, val_ds, test_ds = tokenize_datasets(tokenizer, train_df, val_df, test_df, max_length=MAX_LENGTH)
trainer = make_trainer(model, tokenizer, train_ds, val_ds)

print(f"Training {MODEL_NAME}")
start = time.perf_counter()
trainer.train()
train_seconds = time.perf_counter() - start
print(f"Training finished in {train_seconds:.1f} seconds")


## Validation Threshold Tuning and Final Test Metrics

The decision threshold is selected on validation data by maximizing toxic-class F1. The selected threshold is then applied to the test split once for final reporting.


In [ ]:
val_probs = predict_transformer_probs(trainer, val_ds)
test_probs = predict_transformer_probs(trainer, test_ds)

metrics_at_05_val = binary_metrics(val_df["is_toxic"], val_probs, threshold=0.5)
metrics_at_05_test = binary_metrics(test_df["is_toxic"], test_probs, threshold=0.5)
print_metrics("Validation metrics at threshold 0.5", metrics_at_05_val)
print_metrics("Test metrics at threshold 0.5", metrics_at_05_test)


In [ ]:
def tune_threshold(y_true, toxic_probs, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
    rows = []
    for threshold in thresholds:
        metrics = binary_metrics(y_true, toxic_probs, threshold=float(threshold))
        rows.append({"threshold": float(threshold), **metrics})
    table = pd.DataFrame(rows)
    best_row = table.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]).iloc[0]
    return float(best_row["threshold"]), table

selected_threshold, threshold_table = tune_threshold(val_df["is_toxic"], val_probs)
validation_metrics = binary_metrics(val_df["is_toxic"], val_probs, threshold=selected_threshold)
final_test_metrics = binary_metrics(test_df["is_toxic"], test_probs, threshold=selected_threshold)

print(f"Selected threshold: {selected_threshold:.2f}")
print_metrics("Validation metrics at selected threshold", validation_metrics)
print_metrics("Final test metrics at selected threshold", final_test_metrics)
print("\nClassification report on final test split:")
print(classification_report(
    test_df["is_toxic"],
    (test_probs >= selected_threshold).astype(int),
    target_names=[LABEL_MAPPING[0], LABEL_MAPPING[1]],
    zero_division=0,
))

threshold_table.sort_values("f1", ascending=False).head(10)


## Export Production Artifacts

Artifacts are written under `artifacts/`:

- `toxic_speech_model_pt/`: Hugging Face model and tokenizer.
- `toxic_speech_model_onnx/model.onnx`: PyTorch ONNX export with opset 18 using the current Dynamo exporter. The export uses fixed sequence length (`MAX_LENGTH`) and dynamic batch size, which is simpler and more stable for backend inference.
- `toxic_speech_model_onnx_quantized/model.quantized.onnx`: ONNX Runtime dynamic int8 quantized model when conversion succeeds and the quantized candidate stays close to PyTorch probabilities. The notebook normalizes ONNX `Gemm` nodes before quantization so ONNX Runtime shape inference sees weight metadata that matches the rewritten graph. If quantization fails or the candidate drifts too far, the notebook copies the validated ONNX model into the inference directory and records the fallback in `metrics.json` instead of crashing.
- `label_mapping.json`, `threshold.json`, `metrics.json`: backend metadata.

The notebook does not use the legacy TorchScript ONNX exporter. If `onnxscript` is missing, install it through the setup cell before running export.


In [ ]:
for target in [PT_DIR, ONNX_DIR, QUANTIZED_DIR]:
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(PT_DIR))
tokenizer.save_pretrained(str(PT_DIR))
print(f"Saved Hugging Face model and tokenizer to {PT_DIR}")


In [ ]:
class SequenceClassificationOnnxWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        return self.base_model(**kwargs).logits


try:
    import onnxscript  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch's Dynamo ONNX exporter requires onnxscript. "
        "Run the setup cell, then restart the runtime if needed."
    ) from exc

ONNX_MODEL_PATH = ONNX_DIR / "model.onnx"
OPSET_VERSION = 18
base_export_model = trainer.accelerator.unwrap_model(trainer.model) if hasattr(trainer, "accelerator") else trainer.model
export_model = SequenceClassificationOnnxWrapper(base_export_model.to("cpu")).eval()
example_texts = test_df["text"].head(2).tolist() if len(test_df) >= 2 else ["contoh teks biasa", "contoh teks toxic"]
encoded_example = tokenizer(
    example_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

input_names = ["input_ids", "attention_mask"]
example_inputs = [encoded_example["input_ids"], encoded_example["attention_mask"]]
if "token_type_ids" in encoded_example:
    input_names.append("token_type_ids")
    example_inputs.append(encoded_example["token_type_ids"])

output_names = ["logits"]
batch_dim = torch.export.Dim("batch", min=1, max=64)
dynamic_shapes = tuple({0: batch_dim} for _ in input_names)

with torch.no_grad(), warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=".*cache_position.*deprecated.*",
        category=FutureWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message=".*isinstance\\(treespec, LeafSpec\\).*deprecated.*",
        category=FutureWarning,
    )
    onnx_program = torch.onnx.export(
        export_model,
        tuple(example_inputs),
        f=None,
        input_names=input_names,
        output_names=output_names,
        dynamic_shapes=dynamic_shapes,
        opset_version=OPSET_VERSION,
        dynamo=True,
        optimize=True,
        verify=True,
        external_data=False,
    )
onnx_program.save(str(ONNX_MODEL_PATH))
export_method = "torch.onnx.export dynamo=True fixed_sequence_dynamic_batch"

print(f"Exported ONNX model to {ONNX_MODEL_PATH}")
print(f"ONNX export method: {export_method}")
print(f"ONNX inputs: {input_names}")


In [ ]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quant_pre_process, quantize_dynamic
from onnxruntime.quantization.onnx_model import ONNXModel

PREPROCESSED_ONNX_MODEL_PATH = ONNX_DIR / "model.preprocessed.onnx"
QUANTIZATION_READY_ONNX_MODEL_PATH = ONNX_DIR / "model.quantization_ready.onnx"
CANDIDATE_QUANTIZED_MODEL_PATH = QUANTIZED_DIR / "model.quantized.candidate.onnx"
QUANTIZED_MODEL_PATH = QUANTIZED_DIR / "model.quantized.onnx"
QUANTIZATION_MAX_PROB_DIFF = 0.10
quantized_export_status = "not_started"


def sync_initializer_value_info_shapes(onnx_model):
    initializer_dims = {initializer.name: list(initializer.dims) for initializer in onnx_model.graph().initializer}
    for value_info in onnx_model.graph().value_info:
        dims = initializer_dims.get(value_info.name)
        if not dims or not value_info.type.HasField("tensor_type"):
            continue
        tensor_type = value_info.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        tensor_type.shape.ClearField("dim")
        for dim_value in dims:
            tensor_type.shape.dim.add().dim_value = int(dim_value)


def normalize_gemm_for_dynamic_quantization(input_path, output_path):
    onnx_model = ONNXModel(onnx.load(str(input_path)))
    onnx_model.replace_gemm_with_matmul()
    sync_initializer_value_info_shapes(onnx_model)
    onnx.save_model(onnx_model.model, str(output_path))
    return output_path


def onnx_toxic_probs(model_path, texts):
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    input_names = [item.name for item in session.get_inputs()]
    encoded = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    feed = {name: encoded[name] for name in input_names if name in encoded}
    logits = session.run(None, feed)[0]
    return softmax_np(logits)[:, 1]


def pytorch_toxic_probs(texts):
    pt_model = trainer.model.to("cpu").eval()
    encoded = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    with torch.no_grad():
        logits = pt_model(**encoded).logits.numpy()
    return softmax_np(logits)[:, 1]


def max_candidate_probability_difference(candidate_model_path):
    comparison_texts = test_df["text"].head(8).tolist() if len(test_df) else smoke_examples
    pt_probs = pytorch_toxic_probs(comparison_texts)
    candidate_probs = onnx_toxic_probs(candidate_model_path, comparison_texts)
    return float(np.abs(pt_probs - candidate_probs).max())

try:
    quantization_source = normalize_gemm_for_dynamic_quantization(
        ONNX_MODEL_PATH,
        QUANTIZATION_READY_ONNX_MODEL_PATH,
    )
    print(f"Prepared ONNX model for dynamic quantization: {QUANTIZATION_READY_ONNX_MODEL_PATH}")
except Exception as exc:
    quantization_source = ONNX_MODEL_PATH
    print(f"ONNX Gemm normalization skipped; quantizing exported model directly. Reason: {exc}")

try:
    quant_pre_process(
        input_model_path=str(quantization_source),
        output_model_path=str(PREPROCESSED_ONNX_MODEL_PATH),
        skip_optimization=False,
        skip_onnx_shape=False,
        skip_symbolic_shape=False,
    )
    quantization_source = PREPROCESSED_ONNX_MODEL_PATH
    print(f"Pre-processed ONNX model for quantization: {PREPROCESSED_ONNX_MODEL_PATH}")
except Exception as exc:
    print(f"ONNX pre-processing skipped; quantizing exported model directly. Reason: {exc}")

try:
    if CANDIDATE_QUANTIZED_MODEL_PATH.exists():
        CANDIDATE_QUANTIZED_MODEL_PATH.unlink()
    quantize_dynamic(
        model_input=str(quantization_source),
        model_output=str(CANDIDATE_QUANTIZED_MODEL_PATH),
        op_types_to_quantize=["MatMul"],
        weight_type=QuantType.QInt8,
        per_channel=False,
    )
    candidate_max_diff = max_candidate_probability_difference(CANDIDATE_QUANTIZED_MODEL_PATH)
    if candidate_max_diff <= QUANTIZATION_MAX_PROB_DIFF:
        shutil.move(str(CANDIDATE_QUANTIZED_MODEL_PATH), str(QUANTIZED_MODEL_PATH))
        quantized_export_status = f"quantized_int8_dynamic: max_probability_difference={candidate_max_diff:.6f}"
        print(f"Saved quantized ONNX model to {QUANTIZED_MODEL_PATH}")
        print(f"Quantized max probability difference: {candidate_max_diff:.4f}")
    else:
        CANDIDATE_QUANTIZED_MODEL_PATH.unlink(missing_ok=True)
        shutil.copy2(ONNX_MODEL_PATH, QUANTIZED_MODEL_PATH)
        quantized_export_status = (
            "fallback_unquantized_onnx: quantized_candidate_max_probability_difference="
            f"{candidate_max_diff:.6f} exceeds {QUANTIZATION_MAX_PROB_DIFF:.2f}"
        )
        print("ONNX quantized candidate drifted too far; copied validated ONNX model for inference instead.")
        print(f"Quantized max probability difference: {candidate_max_diff:.4f}")
        print(f"Inference model path: {QUANTIZED_MODEL_PATH}")
except Exception as exc:
    CANDIDATE_QUANTIZED_MODEL_PATH.unlink(missing_ok=True)
    shutil.copy2(ONNX_MODEL_PATH, QUANTIZED_MODEL_PATH)
    quantized_export_status = f"fallback_unquantized_onnx: {type(exc).__name__}: {exc}"
    print("ONNX quantization failed; copied validated ONNX model for inference instead.")
    print(f"Reason: {exc}")
    print(f"Inference model path: {QUANTIZED_MODEL_PATH}")

tokenizer.save_pretrained(str(QUANTIZED_DIR))
trainer.model.config.save_pretrained(str(QUANTIZED_DIR))


In [ ]:
def directory_size_mb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total_bytes = sum(item.stat().st_size for item in path.rglob("*") if item.is_file())
    return total_bytes / (1024 ** 2)

label_mapping_payload = {"0": "non_toxic", "1": "toxic", "label2id": LABEL2ID}
threshold_payload = {
    "threshold": float(selected_threshold),
    "objective": "maximize toxic-class F1 on validation split",
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
}
metrics_payload = {
    "seed": SEED,
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "train_seconds": float(train_seconds),
    "split_rows": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "validation_metrics_at_05": metrics_at_05_val,
    "test_metrics_at_05": metrics_at_05_test,
    "selected_threshold": float(selected_threshold),
    "validation_metrics": validation_metrics,
    "test_metrics": final_test_metrics,
    "onnx_export": {"method": export_method, "opset_version": OPSET_VERSION},
    "quantized_export_status": quantized_export_status,
    "package_versions": installed_package_versions(),
    "artifact_sizes_mb": {
        "pytorch": directory_size_mb(PT_DIR),
        "onnx": directory_size_mb(ONNX_DIR),
        "onnx_quantized": directory_size_mb(QUANTIZED_DIR),
    },
}

(ARTIFACT_DIR / "label_mapping.json").write_text(json.dumps(label_mapping_payload, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "threshold.json").write_text(json.dumps(threshold_payload, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

zip_path = shutil.make_archive("toxic_speech_inference_artifacts", "zip", ARTIFACT_DIR)
print(f"Wrote metadata files under {ARTIFACT_DIR}")
print(f"Created archive: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print(f"Colab download helper unavailable or skipped: {exc}")


## Production Inference Helper

The prediction contract is:

```python
{"label": "toxic" | "non_toxic", "score": toxic_probability, "threshold": selected_threshold}
```


In [ ]:
import onnxruntime as ort

with open(ARTIFACT_DIR / "threshold.json", "r", encoding="utf-8") as handle:
    threshold_config = json.load(handle)
INFERENCE_THRESHOLD = float(threshold_config["threshold"])

ort_tokenizer = AutoTokenizer.from_pretrained(str(QUANTIZED_DIR), use_fast=True)
ort_session = ort.InferenceSession(str(QUANTIZED_MODEL_PATH), providers=["CPUExecutionProvider"])
ort_input_names = [item.name for item in ort_session.get_inputs()]
print(f"Loaded ONNX Runtime model: {QUANTIZED_MODEL_PATH}")
print(f"ORT inputs: {ort_input_names}")


def predict_toxic(text: str) -> dict:
    encoded = ort_tokenizer(
        str(text).strip(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    feed = {name: encoded[name] for name in ort_input_names if name in encoded}
    logits = ort_session.run(None, feed)[0]
    toxic_score = float(softmax_np(logits)[0, 1])
    label_id = int(toxic_score >= INFERENCE_THRESHOLD)
    return {
        "label": LABEL_MAPPING[label_id],
        "score": toxic_score,
        "threshold": INFERENCE_THRESHOLD,
    }


def predict_toxic_batch(texts):
    clean_texts = [str(text).strip() for text in texts]
    encoded = ort_tokenizer(
        clean_texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    feed = {name: encoded[name] for name in ort_input_names if name in encoded}
    logits = ort_session.run(None, feed)[0]
    toxic_scores = softmax_np(logits)[:, 1]
    return [
        {"label": LABEL_MAPPING[int(score >= INFERENCE_THRESHOLD)], "score": float(score), "threshold": INFERENCE_THRESHOLD}
        for score in toxic_scores
    ]


In [ ]:
smoke_examples = [
    "Aku suka belajar machine learning hari ini.",
    "Tolong jangan menghina orang lain di komentar.",
    "Dasar bodoh sekali kelakuan kamu.",
    "Diskusinya panas tapi tetap saling menghormati.",
    "Komentar seperti itu tidak pantas dan menyakitkan.",
]

for text, prediction in zip(smoke_examples, predict_toxic_batch(smoke_examples)):
    print({"text": text, **prediction})


In [ ]:
# CPU inference latency benchmark for quantized ONNX.
latency_texts = test_df["text"].head(16).tolist() if len(test_df) else smoke_examples
single_text = latency_texts[0]

for _ in range(3):
    predict_toxic(single_text)

runs = 50
start = time.perf_counter()
for _ in range(runs):
    predict_toxic(single_text)
single_ms = (time.perf_counter() - start) * 1000 / runs

batch_runs = 20
start = time.perf_counter()
for _ in range(batch_runs):
    predict_toxic_batch(latency_texts)
batch_ms = (time.perf_counter() - start) * 1000 / batch_runs

print(f"Single-text latency: {single_ms:.2f} ms")
print(f"Batch latency for {len(latency_texts)} texts: {batch_ms:.2f} ms")


In [ ]:
# Verify ONNX inference probabilities stay close to PyTorch probabilities on a small sample.
comparison_texts = test_df["text"].head(8).tolist() if len(test_df) else smoke_examples

pt_model = trainer.model.to("cpu").eval()
encoded_pt = tokenizer(
    comparison_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
with torch.no_grad():
    pt_logits = pt_model(**encoded_pt).logits.numpy()
pt_probs = softmax_np(pt_logits)[:, 1]
onnx_probs = np.array([row["score"] for row in predict_toxic_batch(comparison_texts)])
abs_diff = np.abs(pt_probs - onnx_probs)
comparison_df = pd.DataFrame({
    "text": comparison_texts,
    "pytorch_toxic_prob": pt_probs,
    "onnx_inference_toxic_prob": onnx_probs,
    "absolute_difference": abs_diff,
})
display(comparison_df)
print(f"Max absolute probability difference: {abs_diff.max():.4f}")
if abs_diff.max() > 0.10:
    print("Warning: ONNX inference predictions differ more than expected. Inspect export and quantization settings before deployment.")


## Minimal Backend Loading Flow

Expected files after unzipping `toxic_speech_inference_artifacts.zip`:

- `toxic_speech_model_onnx_quantized/model.quantized.onnx`
- `toxic_speech_model_onnx_quantized/tokenizer.json` and tokenizer config files
- `threshold.json`
- `label_mapping.json`
- `metrics.json`

Backend inference flow:

1. Load tokenizer with `AutoTokenizer.from_pretrained("toxic_speech_model_onnx_quantized")`.
2. Load ONNX model with `onnxruntime.InferenceSession(..., providers=["CPUExecutionProvider"])`.
3. Tokenize request text with truncation, padding, and the saved `MAX_LENGTH` from `metrics.json`.
4. Run ONNX logits, apply softmax, read toxic-class probability.
5. Compare the toxic probability against `threshold.json`.
6. Return `label`, `score`, and `threshold`.
